# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefelshazly/FlyRankMachineLearning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #4 — The Freshness Multiplier
**Claim (as I read it):** pages not updated for 365+ days that were refreshed within 30 days show about 3.2x health and 57x impressions, so refresh timing is a strong measured lever.

**Q1: Where does the label come from?** **Q1: Where does the label come from?** Growing vs declining is defined by the 30-day vs previous-30-day impression change (±10%), and the outcome in the 3.2x / 57x claim (health, impressions) is also built from impressions. Freshness itself comes from the last-update date. I'd like to understand how the freshness window and the impression windows line up in time, to rule out that pages were refreshed after they already started growing.

**Q2: Does the validation design carry the claim?** The 3.2x / 57x compares refreshed pages against different, non-refreshed pages. It does not compare the same pages before and after. If pages with existing visibility are the ones chosen for refresh, part of the gap could be selection, not the refresh. A before/after on the same pages, or a match on age, topic and starting impressions, would move this from an observed association to decision-support. I appreciate that the paper already flags the 361+ bucket (283:1 with only 1 declining page) as unstable. That is the standard I'd like to apply here too.

### Finding #8 — The Age-Freshness Matrix
**Claim (as I read it):** old content refreshed in the last 30 days (health 44.62) is close to young-fresh content (44.12), which the paper reads as evidence that refresh works.

**Q1: Where does the label come from?** The outcome is the health score (impressions 30 + position 30 + CTR 20 + scroll 20). Impressions and position are already inside it, so the heatmap partly measures visibility, not quality. I'd ask whether a raw metric such as weighted CTR or clicks shows the same pattern.

**Q2: Does the validation design carry the claim?** The heatmap is a cross-section: one snapshot, different pages in each cell, with no cell sizes or confidence intervals shown. The paper's own methodology states that no p-values or CIs are reported. With cell counts and intervals, I think "consistent with refresh helping" would be well supported. "Proof" is a stronger word than a cross-section can carry. The paper already marks the 365+ × 361+ cell as survivor-biased, and the same caution could apply to the other cells.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** random row split, the same client can sit in train and test. Precision 0.722 vs dummy 0.647 (lift +0.075).
**After:** grouped by client (36 train / 10 test clients). Precision 0.694 vs dummy 0.622 (lift +0.072).

**Observed:** the two lifts are nearly the same, so in this setup the random split did not noticeably inflate the result. Grouped is still the honest design for my question, because FlyRank onboards new clients, and I keep it for that reason, not because it changed the number.

**Robustness (5-fold GroupKFold):** precision ranged from 0.586 to 0.825 across folds (mean 0.714, sd 0.095), but the dummy baseline moved with it (0.466 to 0.769). Lift over the dummy was positive in all 5 folds (about +0.04 to +0.12, mean +0.067). My single 10-client test set was one draw from a noisy distribution.

**Directional reading:** the tree shows a small, consistent edge over "flag everything" across client groups. It is a directional decision-support signal, not a precise estimate of future precision.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score
from sklearn.model_selection import train_test_split, GroupKFold

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

def load(m, s):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) AS avg_position_{s},
               AVG(gsc_impressions) AS avg_impressions_{s},
               ROUND(SUM(gsc_clicks)*1.0/NULLIF(SUM(gsc_impressions),0),4) AS ctr_{s},
               AVG(gsc_clicks) AS avg_clicks_{s}
        FROM read_parquet('{rel}/fact_content_daily_performance/month={m}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id""").df()

march, april = load('2026-03','march'), load('2026-04','april')
df = march.merge(april, on=['client_hash_id','content_hash_id'], how='inner')
df['label'] = (df['avg_position_april'] > df['avg_position_march']).astype(int)
feature_cols = ['avg_position_march','avg_impressions_march','ctr_march','avg_clicks_march']
X, y = df[feature_cols], df['label']
print(df.shape, y.mean().round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(158549, 11) 0.647


In [8]:
def run(Xtr, Xte, ytr, yte):
    p = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr, ytr).predict(Xte)
    d = precision_score(yte, np.ones(len(yte), dtype=int))
    return {"accuracy": accuracy_score(yte, p), "precision": precision_score(yte, p), "dummy_precision": d}

# BEFORE: random row split (same client can sit in train AND test)
a = train_test_split(X, y, test_size=0.2, random_state=42)
before = run(a[0], a[1], a[2], a[3])

# AFTER: grouped by client (my Week-5 split)
clients = df['client_hash_id'].unique()
np.random.seed(42); np.random.shuffle(clients)
tr_c = clients[:int(len(clients)*0.8)]
m = df['client_hash_id'].isin(tr_c)
after = run(X[m], X[~m], y[m], y[~m])

# Robustness: 5-fold GroupKFold
folds = [run(X.iloc[t], X.iloc[v], y.iloc[t], y.iloc[v])
         for t, v in GroupKFold(n_splits=5).split(X, y, groups=df['client_hash_id'])]
cv = pd.DataFrame(folds)

res = pd.DataFrame({"random (before)": before, "grouped (after)": after}).T
res["lift_vs_dummy"] = res["precision"] - res["dummy_precision"]
print(res.round(3))
print("\nGroupKFold per fold:\n", cv.round(3))
print("\nmean precision %.3f ± %.3f | mean lift %.3f" % (
    cv.precision.mean(), cv.precision.std(), (cv.precision-cv.dummy_precision).mean()))

                 accuracy  precision  dummy_precision  lift_vs_dummy
random (before)     0.703      0.745            0.649          0.096
grouped (after)     0.681      0.694            0.622          0.072

GroupKFold per fold:
    accuracy  precision  dummy_precision
0     0.728      0.768            0.690
1     0.708      0.825            0.769
2     0.641      0.586            0.466
3     0.654      0.652            0.606
4     0.709      0.740            0.703

mean precision 0.714 ± 0.095 | mean lift 0.067


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Time leakage:** none found. All four features come from March, and no April column is in the feature set (checked in code).

**Label–feature overlap (the main finding):** the label compares April position to March position, and March position is also a feature. The declining rate falls from 92.4% (March position 1-3) to 73.9% (4-10), 60.6% (11-20), 46.7% (21-50) and 28.3% (50+), and the feature-label correlation is -0.332. Pages near the top have little room to improve and averages tend to drift back toward the middle, so part of the tree's "signal" is likely mechanical, not content-driven. This is not time leakage, but it limits what the position split can be read to mean. Measured alone, impressions, CTR and clicks have weak correlation with the label (0.040, 0.003, 0.029).

**Survivor bias:** the inner join keeps only pages with data in both months. 18,189 March pages (about 10%) have no April row, and their median March position was better (7.00 vs 8.88). Pages with no April data (for any reason, including lost visibility) are missing from the label, so the observed decline rate may be shifted. I did not test which way.

**Client concentration:** the largest client holds 17.2% of rows, so a single client can shift results. I did not test how much this drives the fold spread.

**Verdict:** no direct leakage, but the label is partly determined by the feature, so I read the result as a directional decision-support signal.

**Error examples (grouped test set, 10 clients, 20,974 pages):** 5,024 false positives (flagged declining, actually stable or improving) and 1,658 false negatives (missed a real decline).

- **Where it errs:** error rate rises from 8.6% in the top 3 to 18.6% (positions 4-10), 31.6% (11-20) and 45.0% (21-50). In the 21-50 range the tree is close to a coin flip. This is the zone with the most pages, so the model is least reliable where it would be used most.
- **False positives:** many are small improvements (e.g. position ~28.7 to ~26.5) that the tree called "declining". It leans on March position and flags most mid-ranked pages.
- **False negatives:** include some large drops (e.g. ~25.7 to ~39.3). A few missed pages had only 1-2 impressions in March, but the error rate for pages under 5 impressions (34.3%) is only slightly above the rest (31.0%), so low-impression noise does not explain most errors in this sample.
- **Behavior:** the tree flags 78.2% of test pages as declining (62.2% actually declined) and catches 87.3% of real declines. It behaves close to "flag most pages", with a modest edge on top.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Any April column in features?", any('april' in c for c in feature_cols))
print("\nFeature-label correlation:\n", df[feature_cols+['label']].corr()['label'].round(3))
print("\nLargest client share of rows: %.3f" % df.client_hash_id.value_counts(normalize=True).iloc[0])

# Regression to the mean: label rate by March position bucket
df['pos_bucket'] = pd.cut(df.avg_position_march, [0,3,10,20,50,1000])
print("\nShare declining by March position bucket:\n", df.groupby('pos_bucket', observed=True).label.agg(['mean','count']).round(3))

# Survivor bias: March pages that dropped out of April vs kept
kept = march.merge(april, on=['client_hash_id','content_hash_id'])[['client_hash_id','content_hash_id']]
mm = march.merge(kept.assign(k=1), how='left', on=['client_hash_id','content_hash_id'])
print("\nMarch avg position, kept vs dropped:\n", mm.groupby(mm.k.fillna(0)).avg_position_march.median().round(2))


Any April column in features? False

Feature-label correlation:
 avg_position_march      -0.332
avg_impressions_march    0.040
ctr_march                0.003
avg_clicks_march         0.029
label                    1.000
Name: label, dtype: float64

Largest client share of rows: 0.172

Share declining by March position bucket:
              mean  count
pos_bucket              
(0, 3]      0.924  13583
(3, 10]     0.739  71966
(10, 20]    0.606  30309
(20, 50]    0.467  31624
(50, 1000]  0.283  10308

March avg position, kept vs dropped:
 k
0.0    7.00
1.0    8.88
Name: avg_position_march, dtype: float64


In [10]:
clf = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X[m], y[m])
t = df[~m].copy(); t['pred'] = clf.predict(X[~m])
fp = t[(t.pred==1)&(t.label==0)]   # flagged declining, actually stable/improving
fn = t[(t.pred==0)&(t.label==1)]   # missed a real decline
cols = feature_cols+['avg_position_april']
print("False positives:", len(fp), "| False negatives:", len(fn))
print("\nFP examples:\n", fp[cols].sample(5, random_state=1).round(3).to_string(index=False))
print("\nFN examples:\n", fn[cols].sample(5, random_state=1).round(3).to_string(index=False))
t['err'] = (t.pred!=t.label)
print("\nError rate by March position bucket:\n", t.groupby('pos_bucket', observed=True).err.agg(['mean','count']).round(3))

False positives: 5024 | False negatives: 1658

FP examples:
  avg_position_march  avg_impressions_march  ctr_march  avg_clicks_march  avg_position_april
              5.500                  2.000      0.000             0.000               0.000
             28.664                 13.548      0.000             0.000              26.513
             33.665               1128.742      0.001             0.677              30.468
             32.911                 20.677      0.000             0.000              27.569
             11.083                  2.667      0.000             0.000               5.000

FN examples:
  avg_position_march  avg_impressions_march  ctr_march  avg_clicks_march  avg_position_april
             18.000                  1.000        0.0               0.0              22.000
             53.992                  2.875        0.0               0.0              91.833
             56.333                  1.000        0.0               0.0              73.115
    

In [11]:
print("share flagged:", (t.pred==1).mean().round(3),
      "| recall:", round(((t.pred==1)&(t.label==1)).sum()/(t.label==1).sum(), 3))
print(t.groupby(t.avg_impressions_march<5).err.agg(['mean','count']).round(3))

share flagged: 0.782 | recall: 0.873
                        mean  count
avg_impressions_march              
False                  0.310  15612
True                   0.343   5362


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week 5):** "A 7.2 percentage point improvement... showing the model learned real signal from March GSC data rather than just predicting the majority class."

**Why it goes too far:** it rests on one split of 10 clients, and "learned real signal" is stronger than the evidence. The tree flags 78% of test pages as declining (62% actually declined), so it behaves close to the majority-class strategy. Its errors reach 45% at positions 21-50, and part of its main signal (March position) is built into the label.

**Rewrite:** Observed on a held-out set of 10 clients, the tree's precision was 0.694 against 0.622 for flagging every page (lift +0.072). Across 5 client-grouped folds the measured lift stayed positive (about +0.04 to +0.12), though precision varied from 0.586 to 0.825. This is a directional, decision-support signal for ordering a review queue, not a measured estimate of future accuracy. The model flags most pages and is least reliable at positions 21-50.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

lift = cv.precision - cv.dummy_precision
print("fold lifts:", lift.round(3).tolist(), "| min %.3f max %.3f | all positive: %s" % (lift.min(), lift.max(), (lift>0).all()))
print("share flagged: %.3f | actually declining: %.3f" % ((t.pred==1).mean(), t.label.mean()))


fold lifts: [0.078, 0.056, 0.12, 0.046, 0.036] | min 0.036 max 0.120 | all positive: True
share flagged: 0.782 | actually declining: 0.622


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.